# A Perfect Score on an Empty Mask

**Topic:** Segmentation metrics &nbsp;·&nbsp; **Difficulty:** Medium

Dice and IoU are both `0/0` when the ground-truth mask is empty and the
prediction is empty too. Whether your evaluation scores that case 1.0, 0.0, or
leaves it out of the average changes the headline number by several points on
any medical dataset, where most slices contain nothing at all.

There is no universally right convention. There is only stating which one you
used. Implement all three and measure the spread.

## The data (created in the next cell)

- `gt` `(200, 64, 64)`: binary masks, and 120 of them are entirely empty.
- `pred` `(200, 64, 64)`: predictions, mostly correct, sometimes firing on empty
  slices.

## Your task

With **NumPy only**:

1. Per slice compute `intersection`, `|pred|`, `|gt|`.
2. Dice is `2I / (|p| + |g|)`, IoU is `I / (|p| + |g| - I)`.
3. Handle the empty-empty case three ways and average over all 200 slices:
   `"dice_ones"` (score it 1.0), `"dice_zeros"` (score it 0.0),
   `"dice_skip"` (exclude those slices from the mean).
4. Also report `"iou_skip"` and the counts `"n_empty_gt"`,
   `"n_empty_both"`.
5. Set `answers` to a dict with those six values.

## Evaluation

Compared to the reference to a relative tolerance of 1e-6. The evaluation cell
checks the three conventions genuinely differ.

In [ ]:
import numpy as np

rng = np.random.default_rng(3016)
n, H, W = 200, 64, 64
gt = np.zeros((n, H, W), dtype=bool)
pred = np.zeros((n, H, W), dtype=bool)

yy, xx = np.mgrid[0:H, 0:W]
for i in range(n):
    has_object = i >= 120
    if has_object:
        cy, cx = rng.uniform(18, 46, 2)
        r = rng.uniform(6, 14)
        gt[i] = (yy - cy) ** 2 + (xx - cx) ** 2 < r ** 2
        dy, dx = rng.normal(0, 1.6, 2)
        dr = r * rng.uniform(0.85, 1.15)
        pred[i] = (yy - cy - dy) ** 2 + (xx - cx - dx) ** 2 < dr ** 2
    elif rng.random() < 0.25:
        cy, cx = rng.uniform(18, 46, 2)
        pred[i] = (yy - cy) ** 2 + (xx - cx) ** 2 < 3.0 ** 2
print(f"{n} slices, {int((~gt.any(axis=(1,2))).sum())} with empty ground truth")

In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.

In [ ]:
# =====================================================================
# EVALUATION: run this cell to score your answer
# =====================================================================
import numpy as np

if "answers" not in dir():
    raise NameError("Define `answers` in the solution cell before running this cell.")
if not isinstance(answers, dict):
    raise TypeError("`answers` must be a dict.")

def _digest(a):
    a = np.asarray(a, dtype=float).ravel()
    if a.size == 0:
        return 0.0
    probe = np.random.default_rng(9176).standard_normal(a.size)
    return float(a @ probe)

_CHECKS = [
    ("dice_ones", [], 0.7701084103503711, -0.12894542342101584, 1e-06),
    ("dice_zeros", [], 0.34010841035037115, -0.056947077050781204, 1e-06),
    ("dice_skip", [], 0.5966814216673179, -0.09990715272066879, 1e-06),
    ("iou_skip", [], 0.5234296435197797, -0.08764202040601289, 1e-06),
    ("n_empty_gt", [], 120.0, -20.0925617777399, 1e-06),
    ("n_empty_both", [], 86.0, -14.399669274046929, 1e-06),
]

print("=" * 56)
_ok = True
for _key, _shape, _sum, _probe, _tol in _CHECKS:
    if _key not in answers:
        print(f"  {_key:<24} MISSING")
        _ok = False
        continue
    _a = np.asarray(answers[_key], dtype=float)
    if list(_a.shape) != _shape:
        print(f"  {_key:<24} shape {list(_a.shape)}, expected {_shape}")
        _ok = False
        continue
    if not np.all(np.isfinite(_a)):
        print(f"  {_key:<24} contains NaN or inf")
        _ok = False
        continue
    _g = (abs(_a.sum() - _sum) <= _tol * (1.0 + abs(_sum))
          and abs(_digest(_a) - _probe) <= _tol * (1.0 + abs(_probe)))
    print(f"  {_key:<24} {'ok' if _g else 'wrong'}")
    _ok = _ok and _g

if "dice_ones" in answers:
    _spread = (answers["dice_ones"] - answers["dice_zeros"]) > 0.2
    _order = answers["dice_ones"] > answers["dice_skip"] > answers["dice_zeros"]
    print(f"  {'conventions differ':<24} {'ok' if _spread else 'wrong'}  "
          f"(ones {answers['dice_ones']:.3f}, skip {answers['dice_skip']:.3f}, "
          f"zeros {answers['dice_zeros']:.3f})")
    _ok = _ok and _spread and _order

print("=" * 56)
if _ok:
    print("PASSED, nice work!")
else:
    print("NOT PASSED yet, revisit your solution and try again.")

## Solution

Scoring empty-empty as 1.0 rewards a model for correctly predicting nothing,
which is defensible and makes a do-nothing model score 0.6 on this dataset.
Scoring it 0.0 punishes the same behaviour and is clearly wrong. Skipping is a
common convention in the literature and reports only the slices where there
was something to find, which is why medical segmentation papers must state the
convention, and why comparing numbers across papers that do not is meaningless.

```python
import numpy as np

g = gt.reshape(len(gt), -1)
p = pred.reshape(len(pred), -1)

inter = (g & p).sum(axis=1).astype(float)
sg = g.sum(axis=1).astype(float)
sp = p.sum(axis=1).astype(float)

empty_both = (sg == 0) & (sp == 0)

denom_d = sp + sg
dice = np.divide(2 * inter, denom_d, out=np.zeros_like(inter),
                 where=denom_d > 0)
denom_i = sp + sg - inter
iou = np.divide(inter, denom_i, out=np.zeros_like(inter), where=denom_i > 0)

dice_ones = dice.copy()
dice_ones[empty_both] = 1.0
dice_zeros = dice.copy()
dice_zeros[empty_both] = 0.0

answers = {
    "dice_ones": float(dice_ones.mean()),
    "dice_zeros": float(dice_zeros.mean()),
    "dice_skip": float(dice[~empty_both].mean()),
    "iou_skip": float(iou[~empty_both].mean()),
    "n_empty_gt": int((sg == 0).sum()),
    "n_empty_both": int(empty_both.sum()),
}
```